In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import string
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [3]:
data = pd.read_csv('Emails.csv')

# Removing 'Subject' prefix often found in these datasets
data['text'] = data['text'].str.replace('Subject', '')

# Remove punctuation and stopwords
def clean_text(text):
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = [word.lower() for word in text.split() if word.lower() not in stopwords.words('english')]
    return " ".join(text)

data['text'] = data['text'].apply(clean_text)

In [4]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2)

# Convert labels to numbers (Spam = 1, Ham = 0)
y_train = (y_train == 'spam').astype(int)
y_test = (y_test == 'spam').astype(int)

tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

# Pad sequences so they are all the same length
train_sequences = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=100, padding='post')
test_sequences = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=100, padding='post')

In [5]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Embedding(10000, 32, input_length=100),
    tf.keras.layers.LSTM(16),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid') # Sigmoid for binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [6]:
history = model.fit(train_sequences, y_train, epochs=10, validation_data=(test_sequences, y_test), batch_size=32)

# Check Accuracy
loss, accuracy = model.evaluate(test_sequences, y_test)
print(f"Model Accuracy: {accuracy*100:.2f}%")

Epoch 1/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 4s 21ms/step - accuracy: 0.7314 - loss: 0.5976 - val_accuracy: 0.8580 - val_loss: 0.3131
Epoch 2/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step - accuracy: 0.9051 - loss: 0.2379 - val_accuracy: 0.9150 - val_loss: 0.2879
Epoch 3/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9256 - loss: 0.2605 - val_accuracy: 0.9285 - val_loss: 0.2546
Epoch 4/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9520 - loss: 0.1769 - val_accuracy: 0.9353 - val_loss: 0.2095
Epoch 5/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9410 - loss: 0.1805 - val_accuracy: 0.9401 - val_loss: 0.2578
Epoch 6/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.9647 - loss: 0.1522 - val_accuracy: 0.9401 - val_loss: 0.2198
Epoch 7/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.9552 - loss: 0.1641 - val_accuracy: 0.9411 - val_loss: 0.2092
Epoch 8/10
130/130 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - accuracy: 0.9569 - loss: 0.1491 - val_accu